# MindPulse: Combined Clinical + GoEmotions Training

**Strategy:** Merges your original `mental_training_data.csv` (clinical dataset) with Google Research GoEmotions (social media) for a robust, diverse model.

### Steps:
1. Upload `mental_training_data.csv` to Colab (drag into left sidebar)
2. Run all cells in order
3. Download the new `best_hybrid_model.pt`
4. Test predictions — only replace backend model if better than current

In [ ]:
# STEP 0: Install dependencies
!pip install -q transformers datasets torch pandas scikit-learn nrclex tqdm

In [ ]:
import re
import json
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {DEVICE}")

EMOTION_LABELS = ["Happy/Positive", "Neutral", "Anxious/Stress", "Depressed/Sad"]
EMOTION_LABEL2ID = {label: idx for idx, label in enumerate(EMOTION_LABELS)}

# GoEmotions 27 -> 4 class mapping
GOEMOTIONS_MAPPING = {
    "admiration": "Happy/Positive", "amusement": "Happy/Positive", "approval": "Happy/Positive",
    "caring": "Happy/Positive", "desire": "Happy/Positive", "excitement": "Happy/Positive",
    "gratitude": "Happy/Positive", "joy": "Happy/Positive", "love": "Happy/Positive",
    "optimism": "Happy/Positive", "pride": "Happy/Positive", "relief": "Happy/Positive",
    "neutral": "Neutral", "curiosity": "Neutral", "realization": "Neutral", "surprise": "Neutral",
    "fear": "Anxious/Stress", "nervousness": "Anxious/Stress", "confusion": "Anxious/Stress",
    "anger": "Anxious/Stress", "annoyance": "Anxious/Stress", "disapproval": "Anxious/Stress",
    "sadness": "Depressed/Sad", "disappointment": "Depressed/Sad", "embarrassment": "Depressed/Sad",
    "grief": "Depressed/Sad", "remorse": "Depressed/Sad", "disgust": "Depressed/Sad",
}

def clean_text(text: str) -> str:
    if not isinstance(text, str): return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'([!?.,])\1+', r'\1', text)
    return re.sub(r'\s+', ' ', text).strip()

In [ ]:
# STEP 1: Load your original clinical dataset
# Upload mental_training_data.csv to Colab first (drag into left sidebar)
CLINICAL_CSV = "mental_training_data.csv"

df_clinical = pd.read_csv(CLINICAL_CSV)
df_clinical = df_clinical.dropna(subset=["full_text", "emotion_label"]).copy()
df_clinical["cleaned_text"] = df_clinical["full_text"].apply(clean_text)
df_clinical = df_clinical[df_clinical["cleaned_text"].str.split().str.len() >= 3]
df_clinical = df_clinical[df_clinical["emotion_label"].isin(EMOTION_LABELS)]
df_clinical = df_clinical[["cleaned_text", "emotion_label"]].drop_duplicates(subset=["cleaned_text"])
df_clinical["source"] = "clinical"

print(f"✅ Clinical dataset loaded: {len(df_clinical)} rows")
print(df_clinical["emotion_label"].value_counts())

In [ ]:
# STEP 2: Load GoEmotions from HuggingFace
# Take up to 1500 samples per class so clinical data stays dominant
print("📥 Loading Google GoEmotions...")
dataset = load_dataset("google-research-datasets/go_emotions", "simplified", split="train")
label_names = dataset.features["labels"].feature.names

records = []
for item in tqdm(dataset, desc="Processing GoEmotions"):
    txt = clean_text(item["text"])
    if len(txt.split()) < 3:
        continue
    for l in item["labels"]:
        raw_lab = label_names[l]
        if raw_lab in GOEMOTIONS_MAPPING:
            records.append({"cleaned_text": txt, "emotion_label": GOEMOTIONS_MAPPING[raw_lab], "source": "goemotions"})
            break

df_go = pd.DataFrame(records).drop_duplicates(subset=["cleaned_text"])

# Take 1500 per class from GoEmotions (clinical data will be larger = stays dominant)
go_balanced = []
for emotion in EMOTION_LABELS:
    sub = df_go[df_go["emotion_label"] == emotion]
    n_take = min(len(sub), 1500)
    go_balanced.append(sub.sample(n=n_take, random_state=42))
df_go_balanced = pd.concat(go_balanced).reset_index(drop=True)

print(f"\n✅ GoEmotions balanced: {len(df_go_balanced)} rows")
print(df_go_balanced["emotion_label"].value_counts())

In [ ]:
# STEP 3: Merge clinical + GoEmotions
df_combined = pd.concat([df_clinical, df_go_balanced], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset=["cleaned_text"]).sample(frac=1.0, random_state=42).reset_index(drop=True)
df_combined["emotion_label_id"] = df_combined["emotion_label"].map(EMOTION_LABEL2ID)

print(f"\n✅ Combined Dataset: {len(df_combined)} total rows")
print("Emotion distribution:")
print(df_combined["emotion_label"].value_counts())
print("\nSource distribution:")
print(df_combined["source"].value_counts())

In [ ]:
# STEP 4: Load RoBERTa + extract features
from transformers import AutoTokenizer, AutoModel
from nrclex import NRCLex

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
roberta_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
roberta_model.eval()

NRC_EMOTIONS = ["fear", "anger", "anticipation", "trust", "surprise",
                "positive", "negative", "sadness", "disgust", "joy"]

def extract_nrc(text: str) -> list:
    if not isinstance(text, str) or not text.strip():
        return [0.0] * 10
    try:
        lex = NRCLex()
        lex.load_raw_text(text)
        s = lex.affect_frequencies
        return [float(s.get(e, 0.0)) for e in NRC_EMOTIONS]
    except:
        return [0.0] * 10

@torch.no_grad()
def get_embeddings(texts, batch_size=32):
    all_emb = []
    for i in tqdm(range(0, len(texts), batch_size), desc="RoBERTa Embeddings"):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
        out = roberta_model(**enc)
        mask = enc['attention_mask'].unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        all_emb.append(pooled.cpu().numpy())
    return np.vstack(all_emb)

texts = df_combined["cleaned_text"].tolist()
y_labels = df_combined["emotion_label_id"].values

print("Extracting features...")
roberta_feats = get_embeddings(texts, batch_size=32)
nrc_feats = np.array([extract_nrc(t) for t in tqdm(texts, desc="NRC Lexicon")])
X_features = np.hstack([roberta_feats, nrc_feats])
print(f"Feature matrix: {X_features.shape}")

In [ ]:
# STEP 5: Split + Train
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

X_train, X_temp, y_train, y_temp = train_test_split(X_features, y_labels, test_size=0.2, random_state=42, stratify=y_labels)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

train_loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long)), batch_size=32, shuffle=True)
val_loader   = DataLoader(TensorDataset(torch.tensor(X_val,   dtype=torch.float32), torch.tensor(y_val,   dtype=torch.long)), batch_size=32)
test_loader  = DataLoader(TensorDataset(torch.tensor(X_test,  dtype=torch.float32), torch.tensor(y_test,  dtype=torch.long)), batch_size=32)

class HybridClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(778, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 4)
        )
    def forward(self, x): return self.net(x)

model = HybridClassifier().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

best_val_loss = float('inf')
save_path = "best_hybrid_model.pt"

for epoch in range(20):
    model.train()
    tr_loss, tr_correct, tr_total = 0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad()
        out = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * bx.size(0)
        tr_correct += out.max(1)[1].eq(by).sum().item()
        tr_total += by.size(0)

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            out = model(bx)
            loss = criterion(out, by)
            val_loss += loss.item() * bx.size(0)
            val_correct += out.max(1)[1].eq(by).sum().item()
            val_total += by.size(0)

    vl = val_loss / val_total
    print(f"Epoch {epoch+1:02d}/20 | Train Acc: {tr_correct/tr_total:.4f} | Val Loss: {vl:.4f} Val Acc: {val_correct/val_total:.4f}")
    if vl < best_val_loss:
        best_val_loss = vl
        torch.save(model.state_dict(), save_path)
        print(f"  --> Best checkpoint saved")

# Final test evaluation
model.load_state_dict(torch.load(save_path))
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for bx, by in test_loader:
        out = model(bx.to(DEVICE))
        all_preds.extend(out.max(1)[1].cpu().numpy())
        all_targets.extend(by.numpy())

print(f"\n🎯 Final Test Accuracy: {accuracy_score(all_targets, all_preds):.4f}")
print(classification_report(all_targets, all_preds, target_names=EMOTION_LABELS))

In [ ]:
# STEP 6: Download the new model
# Only replace your backend model if test accuracy is ABOVE 90%
from google.colab import files
files.download('best_hybrid_model.pt')
print("Downloaded! Only replace your backend model if accuracy > 90%")